#### Bring in Required Libraries

In [12]:
import pandas as pd
import pyodbc
import sqlalchemy.types
from sqlalchemy import create_engine
from sqlalchemy import text
from urllib import parse
from timeit import default_timer as timer

In [13]:
# ODBC Connection for FileMaker
start = timer()
cnxn = pyodbc.connect('DSN=FM Clamp ODBC;UID=FMODBC;PWD=FMODBC')
cnxn.timeout = 60
cursor_fm = cnxn.cursor()
print("FileMaker Connect Time = " + str(round((timer() - start), 3)) + " sec")

FileMaker Connect Time = 9.628 sec


In [14]:
# ODBC Connection for AS400
CONNAS400_PROD = """
Driver={iSeries Access ODBC Driver};
system=10.143.12.10;
Server=AS400;
Database=PROD;
UID=SMY;
PWD=SMY;
"""
dbcnxn = pyodbc.connect(CONNAS400_PROD)
cursor_as400 = dbcnxn.cursor()

In [4]:
# ODBC Connection for AS400
CONNAS400_CCSDTA = """
Driver={iSeries Access ODBC Driver};
system=10.143.12.10;
Server=AS400;
Database=PROD;
UID=SMY;
PWD=SMY;
"""
dbcnxn = pyodbc.connect(CONNAS400_CCSDTA)
cursor_as400 = dbcnxn.cursor()

In [15]:
# SQLAlchemy connection for SQL Server
server = 'tn-sql'
database = 'autodata'
driver = 'ODBC+Driver+17+for+SQL+Server'
user = 'production'
pwd = parse.quote_plus("Auto@matics")
port = '1433'
database_conn = f'mssql+pyodbc://{user}:{pwd}@{server}:{port}/{database}?driver={driver}'
# Make Connection
engine = create_engine(database_conn)
# conn = engine.raw_connection()
conn_sql = engine.connect()

In [6]:
def julian_to_date(julian_date):
    """
    Convert 5-digit Julian date (YYDDD) to yyyy-mm-dd format.

    Format: YYDDD where YY=year (assumes the 2000s), DDD=day of year
    Example: 26001 -> 2026-01-01

    Args:
        julian_date: int or str, 5-digit Julian date

    Returns:
        str: Date in 'YYYY-MM-DD' format
    """

    julian_str = str(int(julian_date)).zfill(5)
    year_digits = int(julian_str[:2])
    day_of_year = int(julian_str[2:])

    # Assume years 00-99 map to 2000-2099
    year = 2000 + year_digits

    # Start from January 1 of that year and add days
    base_date = datetime(year, 1, 1).date()
    target_date = base_date + timedelta(days=day_of_year - 1)

    return target_date.strftime('%Y-%m-%d')

In [3]:
# Define SQL queries
sql_screws = "SELECT * FROM autodata.eng.tblRegScrew"
sql_inv = """
    SELECT
        dbo.tblInventory.EngPartNum as partNum,
        '[' + COALESCE(dbo.tblInventory.Cabinet, '') + ' ' + COALESCE(dbo.tblInventory.Drawer, '') + ' ' +
            CONVERT(VARCHAR, COALESCE(dbo.tblInventory.OnHand, 0)) + ']' AS Status
    FROM
        dbo.tblInventory
"""

try:
    # Ensure connection is valid before executing queries
    assert conn_sql is not None, "Database connection (conn_sql) is not defined or invalid."

    # Read data from SQL queries
    df_scr = pd.read_sql(sql_screws, conn_sql)
    df_inv = pd.read_sql(sql_inv, conn_sql)

    # Confirm required columns exist before proceeding (defensive programming)
    assert 'T_CONE PIN' in df_scr.columns, "Column 'T_CONE PIN' is missing in df_scr."
    assert 'partNum' in df_inv.columns, "Column 'partNum' is missing in df_inv."

    # Perform a lookup using pandas merge
    df_scr = pd.merge(
        df_scr, df_inv[['partNum', 'Status']],
        left_on='T_CONE PIN',
        right_on='partNum',
        how='left'
    )

    # Drop unnecessary key column (if needed)
    if 'partNum' in df_scr.columns:
        df_scr.drop(columns=['partNum'], inplace=True)

    # Convert DataFrame dtypes
    df_scr.fillna('None', inplace=True)
    df_db = df_scr.convert_dtypes()

    # Write merged data back to the SQL table
    df_db.to_sql('tblRegScrStatus', conn_sql, schema='eng', if_exists='replace', index=False)

    # Return or display the final DataFrame
    print(df_db.dtypes)

except Exception as e:
    print(f"Error: {e}")

finally:
    try:
        conn_sql.close()
    except Exception as e:
        print(f"Error while closing connection: {e}")

Screw Part Number                  string
Screw Description                  string
Material Part Number               string
Material Information::Grade        string
Clamp Type                         string
                                    ...  
T_ROLLING T. PLATE STATIORY_2nd    string
T_ROLLING DIE_2nd                  string
T_ROLLING T. PLATE MOVABLE_2nd     string
T_Dwg No                           string
Status                             string
Length: 61, dtype: object


In [7]:
sql_spares = """
    SELECT PROD.FPSPRMAST1.SPH_PART,
        STRIP(PROD.FPSPRMAST1.SPH_ENGPRT),
        STRIP(PROD.FPSPRMAST1.SPH_DESC1),
        STRIP(PROD.FPSPRMAST1.SPH_DESC2),
        STRIP(PROD.FPSPRMAST1.SPH_MFG),
        STRIP(PROD.FPSPRMAST1.SPH_MFGPRT),
        STRIP(PROD.FPSPRMAST2.SPD_CABINT),
        STRIP(PROD.FPSPRMAST2.SPD_DRAWER),
        PROD.FPSPRMAST2.SPD_QOHCUR,
        PROD.FPSPRMAST1.SPH_CURSTD,
        STRIP(PROD.FPSPRMAST2.SPD_REODTE),
        STRIP(PROD.FPSPRMAST2.SPD_USECC),
        STRIP(PROD.FPSPRMAST2.SPD_PURCC),
        STRIP(PROD.FPSPRMAST2.SPD_QREORD)
    FROM PROD.FPSPRMAST1 INNER JOIN PROD.FPSPRMAST2 ON PROD.FPSPRMAST1.SPH_PART = PROD.FPSPRMAST2.SPD_PART
    WHERE (((PROD.FPSPRMAST2.SPD_FACIL)=9))
"""

In [4]:
sql_orders = """
    SELECT CCSDTA.DMFMOMR.MFMOMR01 AS Plant,
        STRIP(CCSDTA.DMFMOMR.MFMOMR02)AS OrderNum,
        STRIP(CCSDTA.DMFMOMR.MFMOMR03)AS PartNumber,
        CCSDTA.DMFMOMR.MFMOMR07 As julian_Start,
        CCSDTA.DMFMOMR.MFMOMR09 As julian_Due,
        CCSDTA.DMFMOMR.MFMOMR0A As julian_Close,
        STRIP(CCSDTA.DMFMOMR.MFMOMR0I)AS Note,
        CCSDTA.DMFMOMR.MFMOMR0A As julian_Close
    FROM CCSDTA.DMFMOMR
    WHERE CCSDTA.DMFMOMR.MFMOMR01 =9)
"""

In [16]:
sql_parts = """
    SELECT Ourpart,"Band A Part Number", "Housing A Part Number",
        "Screw Part Number" AS Screw, "Band Feed from Band data",
        "Ship Diam Max", "Ship Diam Min", "Hex Size", "Band_Thickness", "Band_Width",
        "CameraInspectionRequired", "ScrDrvChk", "Cutout1", "DrawingPrintNumber","Size"
    FROM tbl8Tridon
"""

In [35]:
sql_bands = """
    SELECT "Band Part Number", "Feed Length","CutoutA Tool Number","CutoutB Tool Number","CutoutC Tool Number",
        "Dim A", "Dim B","Dim C", "Process", "Description", "Number of Notches", "Abutment Punch", "A Notches Removed",
        "B Notches Removed"
    FROM BANDS
"""

### Execute Spare Part Query on AS400

In [8]:
start = timer()
cursor_as400.execute(sql_spares)
result_spares = cursor_as400.fetchall()
print("AS400 Connect/Query Time = " + str(round((timer() - start), 3)) + " sec")

AS400 Connect/Query Time = 0.776 sec


### Execute Clamp Query on SQL Server

In [17]:
# Clamp Data
start = timer()
cursor_fm.execute(sql_parts)
result_clamp = cursor_fm.fetchall()
print("FileMaker Clamp Query Time = " + str(round((timer() - start), 3)) + " sec")

FileMaker Clamp Query Time = 30.242 sec


### Execute Band Query on FileMaker Server

In [36]:
# Band Data
start = timer()
cursor_fm.execute(sql_bands)
result_band = cursor_fm.fetchall()
print("FileMaker Band Query Time = " + str(round((timer() - start), 3)) + " sec")

FileMaker Band Query Time = 10.168 sec


#### Build Dataframe for Spare Parts

In [ ]:
data_type_dict = {'StandardCost' : float, 'OnHand' : int, 'PartNum' : str, 'ReOrderPt' : int, 'ReOrderDate' : int}
df_spares = pd.DataFrame.from_records(result_spares)
df_spares.columns = ['PartNum', 'EngPartNum', 'Desc1', 'Desc2', 'Mfg', 'MfgPn', 'Cabinet', 'Drawer', 'OnHand', 'StandardCost','ReOrderDate', 'DeptUse', 'DeptPurch', 'ReOrderPt']
df_spares = df_spares.dropna()
df_spares = df_spares.astype(data_type_dict)
df_spares = df_spares.convert_dtypes()

df_obs = df_spares[df_spares.Cabinet.str.contains('OBS')]
df_obs_yest = pd.read_csv('c:\\temp\yesterday_obs.csv',header=None, sep='\t')
df_obs_yest.reset_index(drop=True, inplace=True)
df_obs.reset_index(drop=True, inplace=True)
df_obs_yest = df_obs_yest.fillna("")

df_obs_yest.columns = df_spares.columns
df_obs_yest = df_obs_yest.astype(data_type_dict)
df_obs_yest = df_obs_yest.convert_dtypes()

if not df_obs['PartNum'].equals(df_obs_yest['PartNum']):
    df_diff = pd.concat([df_obs, df_obs_yest]).drop_duplicates(keep=False)
    df_obs.to_csv('c:\\temp\yesterday_obs.csv', header=False, index=False, sep='\t')
    i = 0
    item_list = []
    for index, row in df_diff.iterrows():
        #print(row['PartNum'], row['EngPartNum'], row['Desc1'])
        item_list.append('<h5>Item ' + str(i) + '</h5>'
                             + '<p style="margin-left: 40px">'
                             + 'Part Number: ' + row['PartNum']
                             + '<br>Eng Part Number: <strong>' + row['EngPartNum'] + '</strong>'
                             + '<br>Description 1: ' + row['Desc1']
                             + '<br>Description 2: ' + row['Desc2']
                             + '<br>Manufacturer: ' + row['Mfg']
                             + '<br>Manufacturer Pn: <strong>' + row['MfgPn'] + '</strong>'
                             + '<br>Dept Use: ' + row['DeptUse']
                             + '<br>Dept Purch: ' + row['DeptPurch']
                             + '</p><br>')
        i += 1
df_spares.sample(20) 


#### Build Dataframe for Clamp Data

In [25]:
clamp_columns = [
    'PartNumber', 'Band', 'Housing', 'Screw', 'Feed', 'DiaMax', 'DiaMin', 'HexSz',
    'BandThickness', 'BandWidth', 'CamInspect', 'ScrDrvChk', 'Cutout1', 'Drawing', 'Size'
]

clamp_dtype_map = {
    'PartNumber': str,
    'Band': str,
    'Housing': str,
    'Screw': str,
    'Feed': float,
    'DiaMax': float,
    'DiaMin': float,
    'HexSz': str,
    'BandThickness': float,
    'BandWidth': float,
    'CamInspect': str,
    'ScrDrvChk': str,
    'Cutout1': float,
    'Drawing': str,
    'Size': str
}

numeric_columns = ['Feed', 'DiaMax', 'DiaMin', 'BandThickness', 'BandWidth', 'Cutout1']
uppercase_columns = ['CamInspect', 'ScrDrvChk']
numeric_defaults = {'DiaMax': 0.0, 'DiaMin': 0.0, 'Cutout1': 0.0}

#df_clamps = pd.DataFrame.from_records(result_clamp, columns=clamp_columns)

#clamp_records = [tuple(row) for row in result_clamp]
clamp_records = [
    tuple(row[i] for i in range(len(row)))
    for row in result_clamp
]

df_clamps = pd.DataFrame.from_records(
    clamp_records,
    columns=clamp_columns
)

# Remove FileMaker header/placeholder row and invalid required values
df_clamps = df_clamps.iloc[1:].copy()
df_clamps = df_clamps[df_clamps['Feed'].ne('N/A')]
df_clamps = df_clamps[df_clamps['Size'].ne('None')]

# Clean string columns
string_columns = df_clamps.select_dtypes(include=['object']).columns
df_clamps[string_columns] = df_clamps[string_columns].apply(lambda col: col.astype(str).str.strip())

# Normalize text values
df_clamps[uppercase_columns] = df_clamps[uppercase_columns].apply(lambda col: col.str.upper())
df_clamps['HexSz'] = df_clamps['HexSz'].replace({'Purchased 5/16"': '5/16"'})

# Convert and round numeric columns safely
df_clamps[numeric_columns] = df_clamps[numeric_columns].apply(pd.to_numeric, errors='coerce')
df_clamps.fillna(numeric_defaults, inplace=True)
df_clamps[numeric_columns] = df_clamps[numeric_columns].round(3)

# Final cleanup
df_clamps = df_clamps.dropna()
df_clamps = df_clamps.astype(clamp_dtype_map).convert_dtypes()
df_clamps.drop_duplicates(subset='PartNumber', keep='first', inplace=True)
df_clamps.sort_values(by='PartNumber', inplace=True)

df_clamps.sample(min(20, len(df_clamps)))


,PartNumber,Band,Housing,Screw,Feed,DiaMax,DiaMin,HexSz,BandThickness,BandWidth,CamInspect,ScrDrvChk,Cutout1,Drawing,Size
5016,704053004,2292053,2360007,1299001MC,13.0,3.76,3.68,7 mm,0.025,0.5,NO,NO,5.887,704-001,053
10584,3502700350CT,2662035,2360013,D1300014MC,9.39,2.44,2.32,7 mm,0.025,0.5,NO,NO,0.0,35027 XXXX,035
9915,6223A02,74H0801,1151010,2168001,30.12,0.0,0.0,"3/8""",0.028,0.562,NO,NO,0.572,6200A-01,140
7688,670040032127,2345032,2351045,1318001,8.8,0.0,2.32,"5/16""",0.022,0.5,NO,NO,0.0,67004_67-4,032
1092,665042102,2147042,2157003,2173001,10.864,0.0,0.0,"5/16""",0.024,0.5,NO,NO,0.0,665-001_67-6,042
5206,360140042001,2373042,1151006,1278001MC,10.72,0.0,0.0,"5/16""",0.028,0.562,NO,NO,0.0,3601400XX001,042
10473,380180300001,2958300,7009900,2547014,10.91,2.95,2.79,"5/16""",0.031,0.62,NO,NO,0.0,38018 XXXX,None
836,600056106,2353056,2350016,1213002,13.54,0.0,0.0,"5/16""",0.022,0.5,NO,NO,0.0,600-001_52,056
1863,M229096106,2306096,1019032,D1278001,21.32,0.0,0.0,"5/16""",0.022,0.562,NO,NO,0.0,262-001,096
6842,644P28223,2943028,2360007,1250010MC,8.02,0.0,0.0,"5/16""",0.022,0.5,NO,NO,0.0,644-P01,P28


In [ ]:
df_fixed = df_clamps.sample(min(20, len(df_clamps)))
import numpy as np

# Function to calculate boundaries for outliers using 1.5 × IQR rule
def calculate_boundaries(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return lower_bound, upper_bound

# Function to fix outliers by capping them to the nearest boundary
def fix_outliers(df, col):
    lower_bound, upper_bound = calculate_boundaries(df[col])
    df[col] = df[col].clip(lower=lower_bound, upper=upper_bound)

# Fix outliers for the specified columns
columns_with_outliers = ['Feed', 'DiaMax', 'DiaMin', 'BandThickness', 'BandWidth', 'Cutout1']
for col in columns_with_outliers:
    fix_outliers(df_fixed, col)
df_fixed

In [21]:
df_fixed = df_fixed
def drop_constant_columns(df, constant_columns):
    """
    Function to drop columns with constant values from the dataframe.

    Args:
    df (pd.DataFrame): The dataframe to process.
    constant_columns (list): List of column names to drop because they have constant values.

    Returns:
    None: Modifies the dataframe in place.
    """
    df.drop(columns=constant_columns, inplace=True)

# Define list of columns with constant values
constant_columns = ['ScrDrvChk', 'Cutout1']

# Drop these columns from the dataframe
drop_constant_columns(df_fixed, constant_columns)
df_fixed

,PartNumber,Band,Housing,Screw,Feed,DiaMax,DiaMin,HexSz,BandThickness,BandWidth,CamInspect,Drawing,Size
9960,630170240052,2306240,1019035,2201001,35.07125,0.0,0.0,"5/16""",0.022,0.562,NO,63017 XXXX,240
4117,M550020708,1208020,1209005,D2375003VB,6.375,0.0,0.0,"5/16""",0.022,0.5,NO,550-001,020
1846,M225044102,2314044,1063008,1278001,11.11,0.0,0.0,"5/16""",0.022,0.562,NO,225-001,044
7241,573130140099,4754140,4756003052,4352012052,18.799,5.59,5.51,7 mm,0.028,0.407,NO,573XX-XXXX,140
32,M688048102,1231048,2367025,1213002,12.033,0.0,0.0,"5/16""",0.022,0.5,NO,688-001,048
1577,650049175,2293049,2367025,1278001MC,12.686,3.66,3.58,"5/16""",0.025,0.5,NO,650-001,049
985,631020708,2345020,2351032,1278001,6.45,0.0,0.0,"5/16""",0.022,0.5,NO,631-001,020
10100,600380200052,7N212,7009900,D2542003,10.19,0.0,0.0,"5/16""",0.028,0.62,NO,60038 XXXX,None
374,820550102,2027550,2321001,1228001,19.348,5.7,5.62,"5/16""",0.028,0.5,NO,820 / M820-001,550
8993,HBCP10780,HBCP10-BAND,N03SSBL,None,23.228,2.625,2.56,None,0.026,0.625,NO,HBCPXX780,None


#### Build Dataframe for Band Data

In [37]:
valid_process = ['IN-LINE SINGLE NOTCH DIE', 'IN-LINE DOUBLE NOTCH DIE', 'IN-LINE BAND STAMPING', 'IN-LINE 105 NOTCH DIE', 'CUT & CURL']
df_bands = pd.DataFrame.from_records(result_band)
df_bands.columns = ['Band', 'FeedLength', 'CutOutA', 'CutOutB', 'CutOutC', 'DimA', 'DimB', 'DimC', 'Process', 'Description','NumNotches', 'AbutmentPunch', 'ANotchesRemoved', 'BNotchesRemoved']
data_type_dict = {'Band' : str, 'FeedLength' : float,'CutOutA' : str,'CutOutB' : str,'CutOutC' : str,'DimA' : float,'DimB' : float,'DimC' : float,'Process' : str, 'Description' : str,'NumNotches' : float,'AbutmentPunch' : float,'ANotchesRemoved' : float,'BNotchesRemoved' : float}
df_bands = df_bands.astype(data_type_dict)
df_bands = df_bands.convert_dtypes()

# Shape Data Set
df_bands = df_bands[1:]
df_bands['Band'] = df_bands['Band'].str.upper()
df_bands['Process'] = df_bands['Process'].str.upper()
df_bands = df_bands[df_bands['Process'].isin(valid_process)]
df_bands = df_bands[df_bands['Band'].str.contains('OBS|X|OLD|PSR|CH|NO ACTIVE') == False]
df_bands['FeedLength'] = df_bands['FeedLength'].round(3)
df_bands.fillna({'CutOutA' : 'NON-MULTI'}, inplace=True)
df_bands.fillna({'DimA' : 0.000}, inplace=True)
df_bands.fillna({'DimB' : 0.000}, inplace=True)
df_bands.fillna({'DimC' : 0.000}, inplace=True)
df_bands.fillna({'NumNotches' : 0.000}, inplace=True)
df_bands.fillna({'AbutmentPunch' : 0.000}, inplace=True)
df_bands.fillna({'ANotchesRemoved' : 0.000}, inplace=True)
df_bands.fillna({'BNotchesRemoved' : 0.000}, inplace=True)
df_bands.drop_duplicates(subset='Band', keep='first', inplace=True)
df_bands = df_bands.dropna(subset='Band')

#df_bands.query('BandLength < 10.000', inplace=True)

df_bands.sample(20)
# df_bands.shape

,Band,FeedLength,CutOutA,CutOutB,CutOutC,DimA,DimB,DimC,Process,Description,NumNotches,AbutmentPunch,ANotchesRemoved,BNotchesRemoved
2146,2825758,14.45,091028398,None,None,5.179,0.0,0.0,IN-LINE SINGLE NOTCH DIE,"Band 1/2"" Side Notch",20,0,0,0
2938,2025016,5.66,None,None,None,5.252,0.0,0.0,CUT & CURL,"Band 1/2"" Setup",45,0,0,0
1460,2017037,9.663,None,None,None,0.329,0.313,5.538,CUT & CURL,Band Micro Setup,35,0,0,0
2352,2861747,11.93,091029117,091028259,091029117,2.402,1.488,0.373,IN-LINE SINGLE NOTCH DIE,"Band 1/2"" Dual Slot W/ RH Side Notch",17,0,0,0
1863,2017019,6.0,None,None,None,4.965,5.229,1.875,CUT & CURL,Band Micro Setup,35,0,0,0
9409,2966562,21.14,None,None,None,0.702,10.458,0.0,CUT & CURL,"5/8"" Band, Lined, Marked with the W4 markings",34,0,5,0
11059,2306852,12.68,None,None,None,0.0,0.0,0.0,IN-LINE DOUBLE NOTCH DIE,"9/16"", 301, .022",68,0,0,0
8910,2784046,13.5,091029420,091028259,091028259,4.878,1.456,8.61,CUT & CURL,ETDLHN=Ext Tang w/ Triangle & Dual Left Hand N...,23,0,0,0
3980,1053010,4.712,None,None,None,3.356,0.0,0.0,CUT & CURL,"Band 1/2"" Setup",15,0,0,0
3256,2353024,7.23,None,None,None,1.162,0.0,0.0,CUT & CURL,"Band 1/2"" Setup",37,0,0,0


#### Load Spare Parts to SQL Server Table

In [165]:
data_type_dict = {'StandardCost' : sqlalchemy.types.FLOAT, 'OnHand' : sqlalchemy.types.INT,'PartNum' : sqlalchemy.types.VARCHAR(255),'ReOrderPt' : sqlalchemy.types.INT, 'EngPartNum' : sqlalchemy.types.VARCHAR(255), 'Desc1' : sqlalchemy.types.VARCHAR(255),'Desc2' : sqlalchemy.types.VARCHAR(255),'Mfg' : sqlalchemy.types.VARCHAR(255),'MfgPn' : sqlalchemy.types.VARCHAR(255), 'Cabinet' : sqlalchemy.types.VARCHAR(255),'Drawer' : sqlalchemy.types.VARCHAR(255),'ReOrderDate' : sqlalchemy.types.VARCHAR(255),'DeptUse' : sqlalchemy.types.VARCHAR(255),'DeptPurch' : sqlalchemy.types.VARCHAR(255)}

df_spares.to_sql('tblSpares', conn_sql, schema='eng', if_exists='replace', index=False, dtype=data_type_dict)

28

#### Load Clamp to SQL Server Table

In [29]:
data_type_dict = {'PartNumber' : sqlalchemy.types.VARCHAR(255), 'Band' : sqlalchemy.types.VARCHAR(255),'Housing' : sqlalchemy.types.VARCHAR(255),'Screw' : sqlalchemy.types.VARCHAR(255), 'Feed' : sqlalchemy.types.Float, 'DiaMax' : sqlalchemy.types.Float,'DiaMin' : sqlalchemy.types.Float,'HexSz' : sqlalchemy.types.VARCHAR(255),'BandThickness' : sqlalchemy.types.Float, 'BandWidth' : sqlalchemy.types.Float,'CamInspect' : sqlalchemy.types.VARCHAR(255),'ScrDrvChk' : sqlalchemy.types.VARCHAR(255), 'Cutout1' : sqlalchemy.types.Float,'Drawing' : sqlalchemy.types.VARCHAR(255),'Size' : sqlalchemy.types.VARCHAR(255)}

df_clamps.to_sql('parts_clamps', conn_sql, schema='production', if_exists='replace', index=False, dtype=data_type_dict)

18

#### Load Band Data to SQL Server Table

In [38]:
data_type_dict = {'Band' : sqlalchemy.types.VARCHAR(255),'FeedLength' : sqlalchemy.types.FLOAT, 'CutOutA' : sqlalchemy.types.VARCHAR(255), 'CutOutB' : sqlalchemy.types.VARCHAR(255), 'CutOutC' : sqlalchemy.types.VARCHAR(255),'DimA' : sqlalchemy.types.FLOAT,'DimB' : sqlalchemy.types.FLOAT,'DimC' : sqlalchemy.types.FLOAT, 'Process' : sqlalchemy.types.VARCHAR(255), 'Description' : sqlalchemy.types.VARCHAR(255),'NumNotches' : sqlalchemy.types.FLOAT,'AbutmentPunch' : sqlalchemy.types.FLOAT,'ANotchesRemoved' : sqlalchemy.types.FLOAT,'BNotchesRemoved' : sqlalchemy.types.FLOAT}

df_bands.to_sql('parts_bands', conn_sql, schema='production', if_exists='replace', index=False, dtype=data_type_dict)

119

In [ ]:
def sync_usage(schema="dbo", src_table="tblUsage1_temp", dst_table="tblUsage1", db_engine=None):
    """
    Insert rows from schema.src_table into schema.dst_table that do not already exist in dst_table.
    Excludes identity, computed, rowversion/timestamp columns, and Id.
    Uses null-safe equality in NOT EXISTS to avoid duplicates with NULLs.
    """
    if db_engine is None:
        raise ValueError("sync_usage requires a configured SQLAlchemy engine")

    print(f"Syncing {schema}.{src_table} into {schema}.{dst_table}...")

    with db_engine.begin() as conn:
        cols_sql = text("""
            WITH cols AS (
                SELECT
                    s.name  AS schema_name,
                    t.name  AS table_name,
                    c.name  AS column_name,
                    c.column_id,
                    c.is_identity,
                    c.is_computed,
                    ty.name AS type_name
                FROM sys.schemas s
                JOIN sys.tables t   ON t.schema_id = s.schema_id
                JOIN sys.columns c  ON c.object_id = t.object_id
                JOIN sys.types ty   ON ty.user_type_id = c.user_type_id
                WHERE s.name = :schema
                  AND t.name IN (:src, :dst)
            )
            SELECT d.column_name
            FROM cols d
            JOIN cols s
              ON s.table_name = :src
             AND d.table_name = :dst
             AND s.schema_name = d.schema_name
             AND s.column_name = d.column_name
            WHERE d.schema_name = :schema
              AND d.is_identity = 0
              AND d.is_computed = 0
              AND d.type_name NOT IN ('timestamp', 'rowversion')
              AND d.column_name <> 'Id'
            ORDER BY d.column_id
        """)

        insert_cols = list(
            conn.execute(
                cols_sql,
                {"schema": schema, "src": src_table, "dst": dst_table},
            ).scalars()
        )
        if not insert_cols:
            print("No insertable common columns found.")

        # Remove duplicate rows from the source based on the selected columns
        col_list = ", ".join(f"[{c}]" for c in insert_cols)

        where_parts = [
            f"(d.[{c}] = t.[{c}] OR (d.[{c}] IS NULL AND t.[{c}] IS NULL))"
            for c in insert_cols
        ]
        where_clause = " AND ".join(where_parts)

        sql = f"""
            INSERT INTO [{schema}].[{dst_table}] ({col_list})
            SELECT DISTINCT {col_list}
            FROM [{schema}].[{src_table}] t
            WHERE NOT EXISTS (
                SELECT 1
                FROM [{schema}].[{dst_table}] d
                WHERE {where_clause}
            )
        """

        conn.exec_driver_sql(sql)
        print(f"Sync complete: inserted new rows from {schema}.{src_table} into {schema}.{dst_table}.")

In [ ]:
sync_usage(schema="dbo", src_table="tblUsage_temp1", dst_table="tblUsage1", db_engine=engine)